In [5]:
!rocm-smi



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       5     0x744b,   13037  27.0°C  13.0W  N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  0%     0%    
============================================== End of ROCm SMI Log ===============================================


In [2]:
!git config --global http.sslVerify false && git clone -b finetune/gemma-gateway-270m https://github.com/Blackx16/contxt.git

fatal: destination path 'contxt' already exists and is not an empty directory.


In [6]:
%cd contxt

/workspace/contxt


/opt/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.0

Looking in indexes: https://download.pytorch.org/whl/rocm6.0
  Using cached torchvision-0.19.1%2Brocm6.0-cp310-cp310-linux_x86_64.whl (65.8 MB)
  Using cached torchaudio-2.4.1%2Brocm6.0-cp310-cp310-linux_x86_64.whl (1.7 MB)
  Using cached torch-2.4.1%2Brocm6.0-cp310-cp310-linux_x86_64.whl (2363.4 MB)
  Using cached pillow-12.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pytorch_triton_rocm-3.0.0-cp310-cp310-linux_x86_64.whl (341.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 50.7 kB/s  0:02:05 eta 0:00:06m
  Attempting uninstall: torchm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [pillow]
    Found existing installation: torch 2.13.0━━━━━━━━━━━━━━━━━ 1/5 [pillow]
    Uninstalling torch-2.13.0:90m╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [torch]
      Successfully uninstalled torch-2.13.0━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torchaudio]5 [torchaudio]]


In [2]:
!pip install "transformers>=4.42" datasets accelerate

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [3]:
!pip install optimum

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [7]:
!python finetune/router/train_router.py

device=cuda  base=distilbert-base-uncased
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
train=1438  PRIVATE=899  SHARED=539  class_weights=[1.333951711654663, 0.7997775077819824]
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/linear.py:117: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:273.)
  return F.linear(input, self.weight, self.bias)
epoch 1/4  loss=0.2552
epoch 2/4  loss=0.0801
epoch 3/4  loss=0.0263
epoch 4/4  loss=0.0191
saved → finetune/router/checkpoint  (next: python finetune/router/export_onnx.py)


In [8]:
!rocm-smi



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       5     0x744b,   13037  28.0°C  18.0W  N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  0%     0%    
============================================== End of ROCm SMI Log ===============================================


In [9]:
!python finetune/router/export_onnx.py && python finetune/router/eval_router.py

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.10/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask
exported → finetune/router/onnx-out
  onnx/model.onnx (fp32) + onnx/model_quantized.onnx (int8, ~67 MB)
  upload this folder to HF, e.g. Blackx16/contxt-gateway-router-onnx
| model | n | json valid | tier acc | PRIVATE recall | PRIVATE prec | sens MAE | latency |
|---|---|---|---|---|---|---|---|
| **encoder (checkpoint)** | 253 | 100% | 97.6% | 99.4% | 96.9% | 0.127 | 56ms |

**A/B:** compare this against the generative track's `finetune/dataset/eval.md`. Ship the one with higher tier-accur

In [4]:
import torch

# 1. Verify the PyTorch build
print(f"PyTorch Version: {torch.__version__}")
# This will print the ROCm version if installed correctly (otherwise None)
print(f"ROCm/HIP Version: {torch.version.hip}")

is_available = torch.cuda.is_available()
print(f"\nGPU available: {is_available}")

if is_available:
    device_count = torch.cuda.device_count()
    print(f"Number of GPUs detected: {device_count}")
    
    for i in range(device_count):
        name = torch.cuda.get_device_name(i)
        props = torch.cuda.get_device_properties(i)
        
        # Fallback if the ROCm driver fails to fetch the string
        if not name or not name.strip():
            name = "Unnamed AMD GPU (ROCm String Lookup Quirk)"
            
        print(f"\n--- Device {i}: {name} ---")
        print(f"Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"Compute Capability (gfx architecture): {props.major}.{props.minor}")
        
        # Quick tensor test to guarantee the pipeline works
        try:
            x = torch.tensor([1.0, 2.0]).to(f"cuda:{i}")
            print("Tensor Test: SUCCESS (Data successfully moved to GPU)")
        except Exception as e:
            print(f"Tensor Test: FAILED ({e})")
else:
    print("No GPU detected.")

PyTorch Version: 2.9.1+gitff65f5b
ROCm/HIP Version: 7.2.53211-e1a6bc5663

GPU available: True
Number of GPUs detected: 1

--- Device 0: Unnamed AMD GPU (ROCm String Lookup Quirk) ---
Total VRAM: 47.98 GB
Compute Capability (gfx architecture): 11.0
Tensor Test: SUCCESS (Data successfully moved to GPU)
